# 03 - CNN desde cero

Entrenamos `SimpleCNN` (definida en `src/models/simple_cnn.py`) sin pesos preentrenados.

Etapas:
1. Setup, config, dataloaders y modelo.
2. Loop de entrenamiento con validacion por epoch.
3. Curvas de loss/accuracy.
4. Evaluacion final en test, checkpoint y W&B (opcional).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.utils.config import load_yaml_config
from src.utils.reproducibility import set_global_seed

import torch
from torch import nn, optim

from src.models.simple_cnn import SimpleCNN
from src.training.engine import evaluate_classification, train_one_epoch
from src.utils.wandb_utils import finish_wandb_run, init_wandb_run

CONFIG_PATH = PROJECT_ROOT / "configs" / "scratch_cnn.yaml"
config = load_yaml_config(CONFIG_PATH)
set_global_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else config["device"])
print("Device:", device)
config

## Data loaders

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets

from src.data.dataset import build_image_transforms, load_imagefolder_datasets

data_root = PROJECT_ROOT / config["data"]["root_dir"]
image_size = config["data"]["image_size"]
batch_size = config["data"]["batch_size"]
num_workers = config["data"].get("num_workers", 0)

train_dataset, val_dataset = load_imagefolder_datasets(
    root_dir=data_root,
    train_subdir=config["data"].get("train_subdir", "train"),
    val_subdir=config["data"].get("val_subdir", "val"),
    image_size=image_size,
)

test_dir = data_root / config["data"].get("test_subdir", "test")
test_transform = build_image_transforms(image_size=image_size)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

class_names = train_dataset.classes
num_classes = len(class_names)
print({"train": len(train_dataset), "val": len(val_dataset), "test": len(test_dataset), "num_classes": num_classes})


## Modelo, criterio y optimizador

In [ ]:
model = SimpleCNN(
    num_classes=num_classes,
    channels=config["model"].get("channels", [16, 32, 64]),
    dropout=config["model"].get("dropout", 0.2),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=config["training"].get("weight_decay", 0.0),
)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))
model

## Loop de entrenamiento

In [ ]:
run = init_wandb_run(
    config=config,
    enabled=config["tracking"].get("use_wandb", False),
    project=config["tracking"]["project"],
    run_name=config["tracking"].get("run_name", config["experiment_name"]),
    tags=config["tracking"].get("tags"),
)

history = []
for epoch in range(1, config["training"]["epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate_classification(model, val_loader, criterion, device)
    entry = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
    }
    history.append(entry)
    if run is not None:
        run.log(entry)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

## Curvas de loss/accuracy

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(epochs, [h["train_accuracy"] for h in history], label="train")
axes[1].plot(epochs, [h["val_accuracy"] for h in history], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## Evaluacion en test y guardado del checkpoint

In [ ]:
test_loss, test_acc = evaluate_classification(model, test_loader, criterion, device)
print({"test_loss": test_loss, "test_accuracy": test_acc})

output_dir = PROJECT_ROOT / config["output"]["artifacts_dir"] / "scratch_cnn"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / config["output"]["checkpoint_name"]
torch.save({
    "state_dict": model.state_dict(),
    "model_type": "simple_cnn",
    "class_names": class_names,
    "image_size": image_size,
    "channels": config["model"].get("channels"),
    "dropout": config["model"].get("dropout"),
}, checkpoint_path)
print("Saved checkpoint to", checkpoint_path)

if run is not None:
    run.log({"test_loss": test_loss, "test_accuracy": test_acc})
    finish_wandb_run(run)